# PyTorch LSTM for 72-step water-level forecasting

The input sequence contains the six lagged water levels. The model predicts the next 72 values directly. Scaling is fitted on the training period only.

In [15]:
# torch is not compatible with python version 3.15, but wetterdienst requires python
# version 3.15 -> use torch environment instead of ds_project
from pathlib import Path
import copy
import random
import re
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_PATH = Path('../data/final/pegel_weather_6_hourly_lags_forecast_72h.parquet')
RANDOM_STATE = 42
HOLDOUT_FRACTION = 0.2
EPOCHS = 10
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 32

def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class WaterLevelLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN_SIZE, output_size=72):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, inputs):
        sequence_output, _ = self.lstm(inputs)
        return self.output(sequence_output[:, -1, :])

def load_lstm_data(data_path, column_patterns):
    """Load selected lagged columns into (samples, lags, features) sequences.

    column_patterns may be one regular expression or a list of regular expressions.
    Every selected column must end in ``_lag_<hours>h``.
    """
    patterns = [column_patterns] if isinstance(column_patterns, str) else list(column_patterns)
    regexes = [re.compile(pattern) for pattern in patterns]
    df = pd.read_parquet(data_path).sort_values('timestamp').reset_index(drop=True)
    selected = [
        column for column in df.columns
        if '_lag_' in column and any(regex.search(column) for regex in regexes)
    ]
    if not selected:
        raise ValueError(f'No columns matched patterns: {patterns!r}.')

    def lag_hours(column):
        match = re.search(r'_lag_(\d+)h$', column)
        if match is None:
            raise ValueError(f'Could not parse lag from {column}.')
        return int(match.group(1))

    lags = sorted({lag_hours(column) for column in selected}, reverse=True)
    feature_names = sorted({column.rsplit('_lag_', 1)[0] for column in selected})
    columns_by_lag = {
        lag: [f'{name}_lag_{lag}h' for name in feature_names]
        for lag in lags
    }
    missing = {
        lag: [column for column in columns if column not in selected]
        for lag, columns in columns_by_lag.items()
    }
    missing = {lag: columns for lag, columns in missing.items() if columns}
    if missing:
        raise ValueError(f'Selected predictors are not available for every lag: {missing}')

    feature_frame = df[[column for lag in lags for column in columns_by_lag[lag]]].apply(pd.to_numeric, errors='coerce')
    y = np.stack(df['water_levels'].to_numpy())
    if y.ndim != 2 or y.shape[1] != 73:
        raise ValueError(f'Expected water_levels shape (n_rows, 73), got {y.shape}.')
    if not np.allclose(y[:, 0], df['water_level'].to_numpy(), equal_nan=False):
        raise ValueError('The first water_levels value is not the current water_level.')
    y = y[:, 1:].astype(np.float32)
    valid = feature_frame.notna().all(axis=1).to_numpy() & np.isfinite(y).all(axis=1)
    X = np.stack([
        df[columns_by_lag[lag]].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float32)
        for lag in lags
    ], axis=1)
    return X[valid], y[valid], lags, feature_names

def fit_lstm(X_train, y_train, X_validation, y_validation, epochs=EPOCHS):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = WaterLevelLSTM(input_size=X_train.shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_function = nn.MSELoss()
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=BATCH_SIZE, shuffle=False)
    validation_inputs = torch.from_numpy(X_validation).to(device)
    validation_targets = torch.from_numpy(y_validation).to(device)
    best_state, best_loss, history = None, float('inf'), []
    for epoch in range(epochs):
        model.train()
        train_losses = []
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            loss = loss_function(model(inputs), targets)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            validation_loss = loss_function(model(validation_inputs), validation_targets).item()
        history.append({'epoch': epoch + 1, 'train_mse_scaled': np.mean(train_losses), 'validation_mse_scaled': validation_loss})
        if validation_loss < best_loss:
            best_loss, best_state = validation_loss, copy.deepcopy(model.state_dict())
        print(f'{epoch + 1}/{epochs}: validation MSE={validation_loss:.5f}')
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), device

def run_lstm_experiment(X, y, experiment_name):
    holdout = max(1, int(len(X) * HOLDOUT_FRACTION))
    X_train, X_test = X[:-holdout], X[-holdout:]
    y_train, y_test = y[:-holdout], y[-holdout:]
    feature_mean = X_train.mean(axis=(0, 1), keepdims=True)
    feature_std = X_train.std(axis=(0, 1), keepdims=True)
    feature_std[feature_std == 0] = 1.0
    target_mean, target_std = y_train.mean(axis=0), y_train.std(axis=0)
    target_std[target_std == 0] = 1.0
    X_train_scaled = (X_train - feature_mean) / feature_std
    X_test_scaled = (X_test - feature_mean) / feature_std
    y_train_scaled = (y_train - target_mean) / target_std
    y_test_scaled = (y_test - target_mean) / target_std
    model, history, device = fit_lstm(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled)
    model.eval()
    with torch.no_grad():
        scaled_predictions = model(torch.from_numpy(X_test_scaled).to(device)).cpu().numpy()
    predictions = scaled_predictions * target_std + target_mean
    errors = y_test - predictions
    mse = np.mean(errors ** 2)
    metrics = pd.DataFrame([{'model': experiment_name, 'validation_rows': len(y_test), 'mse': mse, 'rmse': np.sqrt(mse), 'mae': np.mean(np.abs(errors))}])
    horizon_metrics = pd.DataFrame({'model': experiment_name, 'horizon': np.arange(1, 73), 'mse': np.mean(errors ** 2, axis=0), 'rmse': np.sqrt(np.mean(errors ** 2, axis=0)), 'mae': np.mean(np.abs(errors), axis=0)})
    display(metrics)
    return model, history, metrics, horizon_metrics, predictions



    


In [16]:
X_water, y_water, water_lags, water_features = load_lstm_data(
    DATA_PATH,
    column_patterns=r'^water_level_lag_\d+h$',
)
print(f'Water-only sequence shape: {X_water.shape}; lags: {water_lags}; features per lag: {len(water_features)}')
water_lstm, water_history, water_metrics, water_horizon_metrics, water_predictions = run_lstm_experiment(
    X_water, y_water, 'PyTorch LSTM - water-level lags only'
)


Water-only sequence shape: (515429, 6, 1); lags: [6, 5, 4, 3, 2, 1]; features per lag: 1
1/10: validation MSE=0.29176
2/10: validation MSE=0.26543
3/10: validation MSE=0.25771
4/10: validation MSE=0.25480
5/10: validation MSE=0.25319
6/10: validation MSE=0.25203
7/10: validation MSE=0.25111
8/10: validation MSE=0.25036
9/10: validation MSE=0.24975
10/10: validation MSE=0.24924


,model,validation_rows,mse,rmse,mae
0,PyTorch LSTM - water-level lags only,103085,399.303253,19.982574,10.923693


In [17]:
X_weather, y_weather, weather_lags, weather_features = load_lstm_data(
    DATA_PATH,
    column_patterns=r'^(?!water_level_lag_).+_lag_\d+h$',
)
print(f'Weather-only sequence shape: {X_weather.shape}; lags: {weather_lags}; features per lag: {len(weather_features)}')
weather_lstm, weather_history, weather_metrics, weather_horizon_metrics, weather_predictions = run_lstm_experiment(
    X_weather, y_weather, 'PyTorch LSTM - weather lags only'
)


Weather-only sequence shape: (180194, 6, 20); lags: [6, 5, 4, 3, 2, 1]; features per lag: 20
1/10: validation MSE=0.94760
2/10: validation MSE=0.89764
3/10: validation MSE=0.87682
4/10: validation MSE=0.86610
5/10: validation MSE=0.85659
6/10: validation MSE=0.84770
7/10: validation MSE=0.84057
8/10: validation MSE=0.83564
9/10: validation MSE=0.83200
10/10: validation MSE=0.82909


,model,validation_rows,mse,rmse,mae
0,PyTorch LSTM - weather lags only,36038,1061.285889,32.577385,20.71666


In [18]:
X_combined, y_combined, combined_lags, combined_features = load_lstm_data(
    DATA_PATH,
    column_patterns=r'.+_lag_\d+h$',
)
print(f'Combined sequence shape: {X_combined.shape}; lags: {combined_lags}; features per lag: {len(combined_features)}')
combined_lstm, combined_history, combined_metrics, combined_horizon_metrics, combined_predictions = run_lstm_experiment(
    X_combined, y_combined, 'PyTorch LSTM - water-level and weather lags'
)


Combined sequence shape: (180188, 6, 21); lags: [6, 5, 4, 3, 2, 1]; features per lag: 21
1/10: validation MSE=0.72217
2/10: validation MSE=0.39204
3/10: validation MSE=0.35227
4/10: validation MSE=0.33446
5/10: validation MSE=0.32369
6/10: validation MSE=0.31696
7/10: validation MSE=0.31270
8/10: validation MSE=0.30993
9/10: validation MSE=0.30820
10/10: validation MSE=0.30695


,model,validation_rows,mse,rmse,mae
0,PyTorch LSTM - water-level and weather lags,36037,392.79837,19.819141,10.529694
